In [1]:
import numpy as np 
import pandas as pd 

In [2]:
balls = pd.read_csv('../data/raw/ipl_ball_by_ball_2008_2026.csv', low_memory=False)
matches = pd.read_csv('../data/raw/ipl_matches_2008_2026.csv')
season_stats = pd.read_csv('../data/raw/ipl_player_season_stats_2008_2026.csv')

In [3]:
balls.head()

,match_id,season,date,venue,innings,is_super_over,batting_team,bowling_team,over,ball,...,team_score,team_wickets,legal_balls_bowled,current_run_rate,target,runs_needed,balls_left,required_run_rate,review_by,review_decision
0,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,False,Kolkata Knight Riders,Royal Challengers Bengaluru,1,1,...,1,0,1,6.0,NaN,NaN,NaN,NaN,NaN,NaN
1,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,False,Kolkata Knight Riders,Royal Challengers Bengaluru,1,2,...,1,0,2,3.0,NaN,NaN,NaN,NaN,NaN,NaN
2,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,False,Kolkata Knight Riders,Royal Challengers Bengaluru,1,3,...,2,0,2,6.0,NaN,NaN,NaN,NaN,NaN,NaN
3,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,False,Kolkata Knight Riders,Royal Challengers Bengaluru,1,3,...,2,0,3,4.0,NaN,NaN,NaN,NaN,NaN,NaN
4,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,False,Kolkata Knight Riders,Royal Challengers Bengaluru,1,4,...,2,0,4,3.0,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
balls.columns

Index(['match_id', 'season', 'date', 'venue', 'innings', 'is_super_over',
       'batting_team', 'bowling_team', 'over', 'ball', 'delivery_in_over',
       'phase', 'batter', 'non_striker', 'bowler', 'batter_runs', 'extra_runs',
       'total_runs', 'extra_type', 'is_legal_ball', 'is_four', 'is_six',
       'is_dot', 'is_wicket', 'player_out', 'dismissal_kind', 'fielders',
       'bowler_wicket', 'team_score', 'team_wickets', 'legal_balls_bowled',
       'current_run_rate', 'target', 'runs_needed', 'balls_left',
       'required_run_rate', 'review_by', 'review_decision'],
      dtype='object')

In [5]:
balls['batter'].value_counts()

batter
V Kohli           7127
RG Sharma         5659
S Dhawan          5483
DA Warner         4849
AM Rahane         4391
                  ... 
NJ Rimmington        1
S Kaushik            1
PVSN Raju            1
V Pratap Singh       1
Sunny Gupta          1
Name: count, Length: 738, dtype: int64

In [6]:
balls.groupby('batter')['batter_runs'].sum().sort_values(ascending=False).sample(10)

batter
Mohammed Siraj      117
N Pooran           2527
SM Katich           241
KV Sharma           352
KK Nair            1712
Vishnu Vinod         71
Harshit Rana         59
SS Prabhudessai     126
PJ Sangwan           26
DAJ Bracewell        12
Name: batter_runs, dtype: int64

In [7]:
from pathlib import Path

# Columns that can contain player names
player_columns = [
    col for col in [
        "batter",
        "non_striker",
        "bowler",
        "player_out",
        "fielder",
        "fielders"
    ]
    if col in balls.columns
]

if not player_columns:
    raise ValueError("No player columns found in the balls DataFrame.")

# Combine all player names
players = pd.concat(
    [balls[col] for col in player_columns],
    ignore_index=True
)

# Handle columns containing multiple fielders
players = (
    players.dropna()
    .astype(str)
    .str.split(r"[,;|]")
    .explode()
    .str.strip()
)

# Remove empty and invalid values
players = players[
    ~players.str.lower().isin(["", "nan", "none", "null", "-"])
]

# Remove duplicates and sort
players = sorted(players.unique(), key=str.lower)

# Save CSV
output_path = Path("../data/processed/ipl_players.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

player_df = pd.DataFrame({
    "player_name": players
})

player_df.to_csv(output_path, index=False)

print(f"Total unique players: {len(player_df)}")
print(f"Saved successfully: {output_path}")

Total unique players: 816
Saved successfully: ..\data\processed\ipl_players.csv


In [8]:
player_info_df  = pd.read_csv("../data/raw/players.csv")

In [9]:
player_info_df.shape

(804, 10)

In [10]:
player_info_df.head(2)

,player_id,cricsheet_registry_id,canonical_name,full_name,nationality,batting_style,bowling_style,playing_role,is_overseas,created_at
0,3543,18e6906e,A Choudhary,Arpita Choudhary,India,Right-hand bat,No information available,Batter,False,2026-04-28 05:21:24.773242+00
1,3545,96fd40ae,A Nehra,Ashish Nehra,India,Right-hand bat,Left-arm fast-medium,Bowler,False,2026-04-28 05:21:24.773242+00


In [11]:
player_info_df[['player_id', 'canonical_name', 'full_name', 'nationality', 'batting_style', 'bowling_style', 'playing_role']]

,player_id,canonical_name,full_name,nationality,batting_style,bowling_style,playing_role
0,3543,A Choudhary,Arpita Choudhary,India,Right-hand bat,No information available,Batter
1,3545,A Nehra,Ashish Nehra,India,Right-hand bat,Left-arm fast-medium,Bowler
2,3547,B Kumar,Bhuvneshwar Kumar,India,Right-hand bat,Right-arm fast-medium,Bowler
3,3548,BCJ Cutting,Ben Cutting,Australia,Right-hand bat,Right-arm fast-medium,All-rounder
4,3549,Bipul Sharma,Bipul Sharma,India,Left-hand bat,Left-arm orthodox,Bowler
...,...,...,...,...,...,...,...
799,35828,PSP Handscomb,Peter Handscomb,Australia,Right-hand bat,No information available,Wicket-keeper batter
800,35941,SM Boland,Scott Boland,Australia,Right-hand bat,Right-arm fast-medium,Bowler
801,36053,UT Khawaja,Usman Khawaja,Australia,Left-hand bat,Right-arm off-spin,Batter
802,36198,F Behardien,No information available,No information available,No information available,No information available,No information available


In [12]:
# Select player information columns
player_columns = [
    "player_id",
    "canonical_name",
    "full_name",
    "nationality",
    "batting_style",
    "bowling_style",
    "playing_role"
]

# Extract player information
player_metadata = player_info_df[player_columns].copy()

# Remove duplicate player IDs
player_metadata = player_metadata.drop_duplicates(
    subset="player_id"
)

# Save CSV
output_path = Path("../data/processed/player_metadata.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

player_metadata.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Total players: {len(player_metadata)}")
print(f"Saved to: {output_path}")


Total players: 804
Saved to: ..\data\processed\player_metadata.csv


In [13]:
player_metadata_df = pd.read_csv('..\data\processed\player_metadata.csv')

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\us\AppData\Local\Temp\ipykernel_19584\2746698074.py:1: SyntaxWarning: invalid escape sequence '\d'
  player_metadata_df = pd.read_csv('..\data\processed\player_metadata.csv')


In [14]:
player_metadata_df

,player_id,canonical_name,full_name,nationality,batting_style,bowling_style,playing_role
0,3543,A Choudhary,Arpita Choudhary,India,Right-hand bat,No information available,Batter
1,3545,A Nehra,Ashish Nehra,India,Right-hand bat,Left-arm fast-medium,Bowler
2,3547,B Kumar,Bhuvneshwar Kumar,India,Right-hand bat,Right-arm fast-medium,Bowler
3,3548,BCJ Cutting,Ben Cutting,Australia,Right-hand bat,Right-arm fast-medium,All-rounder
4,3549,Bipul Sharma,Bipul Sharma,India,Left-hand bat,Left-arm orthodox,Bowler
...,...,...,...,...,...,...,...
799,35828,PSP Handscomb,Peter Handscomb,Australia,Right-hand bat,No information available,Wicket-keeper batter
800,35941,SM Boland,Scott Boland,Australia,Right-hand bat,Right-arm fast-medium,Bowler
801,36053,UT Khawaja,Usman Khawaja,Australia,Left-hand bat,Right-arm off-spin,Batter
802,36198,F Behardien,No information available,No information available,No information available,No information available,No information available


In [15]:
player_metadata_df[['full_name', 'playing_role']].sort_values('playing_role')

,full_name,playing_role
401,Tanay Thyagarajan,All-rounder
503,David Hussey,All-rounder
154,Marlon Samuels,All-rounder
496,Krish Bhagat,All-rounder
493,Mayank Rawat,All-rounder
...,...,...
507,Mark Boucher,Wicket-keeper batter
498,Ravi Singh,Wicket-keeper batter
757,Nitin Saini,Wicket-keeper batter
471,Kusal Mendis,Wicket-keeper batter


In [16]:
temp_df = (
    balls.groupby('batter', as_index=False)['batter_runs']
    .sum()
)

temp_df.head()

,batter,batter_runs
0,A Ashish Reddy,280
1,A Badoni,1178
2,A Chandila,4
3,A Chopra,53
4,A Choudhary,25


In [26]:
temp_df[temp_df['batter'].str.contains("V ")]

,batter,batter_runs
148,CV Varun,31
195,DV Malewar,2
233,HV Patel,280
325,KV Sharma,352
412,MV Boucher,394
447,NV Ojha,1554
486,PV Tambe,18
548,RV Gomez,50
549,RV Patel,80
550,RV Uthappa,4954


In [17]:
temp_df.shape

(738, 2)

In [27]:
player_metadata_df = player_metadata_df.merge(
    temp_df,
    left_on='canonical_name',
    right_on='batter',
    how='left'
)

In [28]:
player_metadata_df.shape

(804, 9)

In [29]:
player_metadata_df

,player_id,canonical_name,full_name,nationality,batting_style,bowling_style,playing_role,batter,batter_runs
0,3543,A Choudhary,Arpita Choudhary,India,Right-hand bat,No information available,Batter,A Choudhary,25.0
1,3545,A Nehra,Ashish Nehra,India,Right-hand bat,Left-arm fast-medium,Bowler,A Nehra,41.0
2,3547,B Kumar,Bhuvneshwar Kumar,India,Right-hand bat,Right-arm fast-medium,Bowler,B Kumar,354.0
3,3548,BCJ Cutting,Ben Cutting,Australia,Right-hand bat,Right-arm fast-medium,All-rounder,BCJ Cutting,238.0
4,3549,Bipul Sharma,Bipul Sharma,India,Left-hand bat,Left-arm orthodox,Bowler,Bipul Sharma,187.0
...,...,...,...,...,...,...,...,...,...
799,35828,PSP Handscomb,Peter Handscomb,Australia,Right-hand bat,No information available,Wicket-keeper batter,PSP Handscomb,6.0
800,35941,SM Boland,Scott Boland,Australia,Right-hand bat,Right-arm fast-medium,Bowler,NaN,NaN
801,36053,UT Khawaja,Usman Khawaja,Australia,Left-hand bat,Right-arm off-spin,Batter,UT Khawaja,127.0
802,36198,F Behardien,No information available,No information available,No information available,No information available,No information available,F Behardien,14.0


In [30]:
temp_df = balls.groupby('bowler', as_index=False)['is_wicket'].sum()
temp_df.sort_values('is_wicket', ascending=False)

,bowler,is_wicket
76,B Kumar,243
569,YS Chahal,242
480,SP Narine,229
213,JJ Bumrah,208
131,DJ Bravo,207
...,...,...
259,Karim Janat,0
251,KR Mayers,0
253,KS Williamson,0
567,YBK Jaiswal,0


In [31]:
player_metadata_df

,player_id,canonical_name,full_name,nationality,batting_style,bowling_style,playing_role,batter,batter_runs
0,3543,A Choudhary,Arpita Choudhary,India,Right-hand bat,No information available,Batter,A Choudhary,25.0
1,3545,A Nehra,Ashish Nehra,India,Right-hand bat,Left-arm fast-medium,Bowler,A Nehra,41.0
2,3547,B Kumar,Bhuvneshwar Kumar,India,Right-hand bat,Right-arm fast-medium,Bowler,B Kumar,354.0
3,3548,BCJ Cutting,Ben Cutting,Australia,Right-hand bat,Right-arm fast-medium,All-rounder,BCJ Cutting,238.0
4,3549,Bipul Sharma,Bipul Sharma,India,Left-hand bat,Left-arm orthodox,Bowler,Bipul Sharma,187.0
...,...,...,...,...,...,...,...,...,...
799,35828,PSP Handscomb,Peter Handscomb,Australia,Right-hand bat,No information available,Wicket-keeper batter,PSP Handscomb,6.0
800,35941,SM Boland,Scott Boland,Australia,Right-hand bat,Right-arm fast-medium,Bowler,NaN,NaN
801,36053,UT Khawaja,Usman Khawaja,Australia,Left-hand bat,Right-arm off-spin,Batter,UT Khawaja,127.0
802,36198,F Behardien,No information available,No information available,No information available,No information available,No information available,F Behardien,14.0


In [32]:
player_metadata_df = player_metadata_df.merge(
    temp_df,
    left_on='canonical_name',
    right_on='bowler',
    how='left'
)

In [33]:
player_metadata_df

,player_id,canonical_name,full_name,nationality,batting_style,bowling_style,playing_role,batter,batter_runs,bowler,is_wicket
0,3543,A Choudhary,Arpita Choudhary,India,Right-hand bat,No information available,Batter,A Choudhary,25.0,A Choudhary,5.0
1,3545,A Nehra,Ashish Nehra,India,Right-hand bat,Left-arm fast-medium,Bowler,A Nehra,41.0,A Nehra,121.0
2,3547,B Kumar,Bhuvneshwar Kumar,India,Right-hand bat,Right-arm fast-medium,Bowler,B Kumar,354.0,B Kumar,243.0
3,3548,BCJ Cutting,Ben Cutting,Australia,Right-hand bat,Right-arm fast-medium,All-rounder,BCJ Cutting,238.0,BCJ Cutting,12.0
4,3549,Bipul Sharma,Bipul Sharma,India,Left-hand bat,Left-arm orthodox,Bowler,Bipul Sharma,187.0,Bipul Sharma,22.0
...,...,...,...,...,...,...,...,...,...,...,...
799,35828,PSP Handscomb,Peter Handscomb,Australia,Right-hand bat,No information available,Wicket-keeper batter,PSP Handscomb,6.0,NaN,NaN
800,35941,SM Boland,Scott Boland,Australia,Right-hand bat,Right-arm fast-medium,Bowler,NaN,NaN,SM Boland,3.0
801,36053,UT Khawaja,Usman Khawaja,Australia,Left-hand bat,Right-arm off-spin,Batter,UT Khawaja,127.0,NaN,NaN
802,36198,F Behardien,No information available,No information available,No information available,No information available,No information available,F Behardien,14.0,NaN,NaN


In [34]:
player_metadata_df.to_csv('../data/processed/player_metadata.csv', index=False, encoding="utf-8-sig")

In [35]:
player_metadata_df = pd.read_csv('..\data\processed\player_metadata.csv')

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\us\AppData\Local\Temp\ipykernel_19584\2746698074.py:1: SyntaxWarning: invalid escape sequence '\d'
  player_metadata_df = pd.read_csv('..\data\processed\player_metadata.csv')


In [36]:
player_metadata_df

,player_id,canonical_name,full_name,nationality,batting_style,bowling_style,playing_role,batter,batter_runs,bowler,is_wicket
0,3543,A Choudhary,Arpita Choudhary,India,Right-hand bat,No information available,Batter,A Choudhary,25.0,A Choudhary,5.0
1,3545,A Nehra,Ashish Nehra,India,Right-hand bat,Left-arm fast-medium,Bowler,A Nehra,41.0,A Nehra,121.0
2,3547,B Kumar,Bhuvneshwar Kumar,India,Right-hand bat,Right-arm fast-medium,Bowler,B Kumar,354.0,B Kumar,243.0
3,3548,BCJ Cutting,Ben Cutting,Australia,Right-hand bat,Right-arm fast-medium,All-rounder,BCJ Cutting,238.0,BCJ Cutting,12.0
4,3549,Bipul Sharma,Bipul Sharma,India,Left-hand bat,Left-arm orthodox,Bowler,Bipul Sharma,187.0,Bipul Sharma,22.0
...,...,...,...,...,...,...,...,...,...,...,...
799,35828,PSP Handscomb,Peter Handscomb,Australia,Right-hand bat,No information available,Wicket-keeper batter,PSP Handscomb,6.0,NaN,NaN
800,35941,SM Boland,Scott Boland,Australia,Right-hand bat,Right-arm fast-medium,Bowler,NaN,NaN,SM Boland,3.0
801,36053,UT Khawaja,Usman Khawaja,Australia,Left-hand bat,Right-arm off-spin,Batter,UT Khawaja,127.0,NaN,NaN
802,36198,F Behardien,No information available,No information available,No information available,No information available,No information available,F Behardien,14.0,NaN,NaN


In [39]:
player_metadata_df.sort_values('batter_runs', ascending=False)['full_name'].unique().shape

(732,)

In [38]:
player_metadata_df.sort_values('is_wicket', ascending=False)['full_name'].unique()

array(['Bhuvneshwar Kumar', 'Yuzvendra Chahal', 'Sunil Narine',
       'Jasprit Bumrah', 'Dwayne Bravo', 'Ravichandran Ashwin',
       'Piyush Chawla', 'Ravindra Jadeja', 'Lasith Malinga',
       'Rashid Khan', 'Amit Mishra', 'Sanjeev Sharma', 'Harshal Patel',
       'Umesh Yadav', 'Mohammed Shami', 'Harbhajan Singh',
       'Kagiso Rabada', 'Trent Boult', 'Mohit Sharma', 'Axar Patel',
       'Andre Russell', 'Mohammed Siraj', 'Vinay Kumar', 'Shardul Thakur',
       'Jaydev Unadkat', 'Ashish Nehra', 'Arshdeep Singh', 'Zaheer Khan',
       'Kuldeep Yadav', 'Varun Chakaravarthy', 'No information available',
       'Chris Morris', 'Shane Watson', 'Dale Steyn', 'Praveen Kumar',
       'R P Singh', 'Ishant Sharma', 'Avesh Khan', 'Pragyan Ojha',
       'Deepak Chahar', 'Irfan Pathan', 'Prasidh Krishna', 'Albie Morkel',
       'Khaleel Ahmed', 'Dhawal Kulkarni', 'Mitchell Starc',
       'Pat Cummins', 'Jofra Archer', 'Karan Sharma', 'Ravi Bishnoi',
       'Morne Morkel', 'Hardik Pandya', 'Imr

In [44]:
batters = player_metadata_df.dropna(subset=["full_name"]).sort_values("batter_runs", ascending=False).drop_duplicates(subset="canonical_name").query("full_name != 'No information available'")["full_name"].tolist()

In [45]:
len(batters)

739

In [54]:
player_metadata_df = pd.read_csv('..\data\processed\player_metadata.csv')

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\us\AppData\Local\Temp\ipykernel_19584\2746698074.py:1: SyntaxWarning: invalid escape sequence '\d'
  player_metadata_df = pd.read_csv('..\data\processed\player_metadata.csv')


In [55]:
player_metadata_df[player_metadata_df['full_name'].str.contains('No information available')]

,player_id,canonical_name,full_name,nationality,batting_style,bowling_style,playing_role,batter,batter_runs,bowler,is_wicket


In [61]:
player_metadata_df.columns

Index(['player_id', 'canonical_name', 'full_name', 'nationality',
       'batting_style', 'bowling_style', 'playing_role', 'batter',
       'batter_runs', 'bowler', 'is_wicket'],
      dtype='object')

In [60]:
balls.columns

Index(['match_id', 'season', 'date', 'venue', 'innings', 'is_super_over',
       'batting_team', 'bowling_team', 'over', 'ball', 'delivery_in_over',
       'phase', 'batter', 'non_striker', 'bowler', 'batter_runs', 'extra_runs',
       'total_runs', 'extra_type', 'is_legal_ball', 'is_four', 'is_six',
       'is_dot', 'is_wicket', 'player_out', 'dismissal_kind', 'fielders',
       'bowler_wicket', 'team_score', 'team_wickets', 'legal_balls_bowled',
       'current_run_rate', 'target', 'runs_needed', 'balls_left',
       'required_run_rate', 'review_by', 'review_decision'],
      dtype='object')

In [59]:
balls[balls['season'] == 2026]

,match_id,season,date,venue,innings,is_super_over,batting_team,bowling_team,over,ball,...,team_score,team_wickets,legal_balls_bowled,current_run_rate,target,runs_needed,balls_left,required_run_rate,review_by,review_decision
278205,1527674,2026,2026-03-28,M Chinnaswamy Stadium,1,False,Sunrisers Hyderabad,Royal Challengers Bengaluru,1,1,...,0,0,1,0.00,NaN,NaN,NaN,NaN,NaN,NaN
278206,1527674,2026,2026-03-28,M Chinnaswamy Stadium,1,False,Sunrisers Hyderabad,Royal Challengers Bengaluru,1,2,...,1,0,2,3.00,NaN,NaN,NaN,NaN,NaN,NaN
278207,1527674,2026,2026-03-28,M Chinnaswamy Stadium,1,False,Sunrisers Hyderabad,Royal Challengers Bengaluru,1,3,...,1,0,3,2.00,NaN,NaN,NaN,NaN,Royal Challengers Bengaluru,struck down
278208,1527674,2026,2026-03-28,M Chinnaswamy Stadium,1,False,Sunrisers Hyderabad,Royal Challengers Bengaluru,1,4,...,7,0,4,10.50,NaN,NaN,NaN,NaN,NaN,NaN
278209,1527674,2026,2026-03-28,M Chinnaswamy Stadium,1,False,Sunrisers Hyderabad,Royal Challengers Bengaluru,1,5,...,7,0,5,8.40,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295727,1535465,2026,2026-05-31,Narendra Modi Stadium (Motera),2,False,Royal Challengers Bengaluru,Gujarat Titans,18,2,...,150,5,104,8.65,156.0,6.0,16.0,2.25,NaN,NaN
295728,1535465,2026,2026-05-31,Narendra Modi Stadium (Motera),2,False,Royal Challengers Bengaluru,Gujarat Titans,18,3,...,150,5,105,8.57,156.0,6.0,15.0,2.40,NaN,NaN
295729,1535465,2026,2026-05-31,Narendra Modi Stadium (Motera),2,False,Royal Challengers Bengaluru,Gujarat Titans,18,4,...,151,5,106,8.55,156.0,5.0,14.0,2.14,NaN,NaN
295730,1535465,2026,2026-05-31,Narendra Modi Stadium (Motera),2,False,Royal Challengers Bengaluru,Gujarat Titans,18,5,...,155,5,107,8.69,156.0,1.0,13.0,0.46,NaN,NaN


In [62]:
# Ensure season and date have correct types
balls['season'] = pd.to_numeric(balls['season'], errors='coerce')
balls['date'] = pd.to_datetime(balls['date'], errors='coerce')

# Players appearing as batters
batting_appearances = balls[
    ['season', 'date', 'batter', 'batting_team']
].rename(columns={
    'batter': 'canonical_name',
    'batting_team': 'team'
})

# Players appearing as bowlers
bowling_appearances = balls[
    ['season', 'date', 'bowler', 'bowling_team']
].rename(columns={
    'bowler': 'canonical_name',
    'bowling_team': 'team'
})

# Combine both appearances
appearances = pd.concat(
    [batting_appearances, bowling_appearances],
    ignore_index=True
)

# Remove invalid records
appearances = appearances.dropna(
    subset=['season', 'canonical_name', 'team']
)

# Sort chronologically
appearances = appearances.sort_values(
    ['season', 'date']
)


In [63]:

# First IPL season
debut_df = (
    appearances.groupby('canonical_name')['season']
    .min()
    .rename('debut')
)

# Last IPL season
last_season_df = (
    appearances.groupby('canonical_name')['season']
    .max()
    .rename('last_season')
)

# Team from the player's latest appearance
latest_team_df = (
    appearances
    .drop_duplicates('canonical_name', keep='last')
    [['canonical_name', 'team']]
)

# Combine player information
career_info = pd.concat(
    [debut_df, last_season_df],
    axis=1
).reset_index()

career_info = career_info.merge(
    latest_team_df,
    on='canonical_name',
    how='left'
)

career_info.head()


,canonical_name,debut,last_season,team
0,A Ashish Reddy,2012,2016,Sunrisers Hyderabad
1,A Badoni,2022,2026,Lucknow Super Giants
2,A Chandila,2012,2013,Rajasthan Royals
3,A Chopra,2008,2009,Kolkata Knight Riders
4,A Choudhary,2017,2017,Royal Challengers Bengaluru


In [67]:
career_info[career_info['canonical_name'].str.contains("V S")]

,canonical_name,debut,last_season,team
352,KV Sharma,2009,2025,Mumbai Indians
685,SV Samson,2013,2026,Chennai Super Kings
760,V Sehwag,2008,2015,Punjab Kings
761,V Shankar,2014,2025,Chennai Super Kings
762,V Suryavanshi,2025,2026,Rajasthan Royals
768,VRV Singh,2008,2010,Punjab Kings


In [68]:
career_info.shape

(806, 4)

In [69]:
# Remove previously generated columns if they exist
player_metadata_df = player_metadata_df.drop(
    columns=['team', 'debut', 'last_season'],
    errors='ignore'
)

# Merge career information
player_metadata_df = player_metadata_df.merge(
    career_info,
    on='canonical_name',
    how='left'
)

# Convert season columns to nullable integers
player_metadata_df['debut'] = (
    player_metadata_df['debut'].astype('Int64')
)

player_metadata_df['last_season'] = (
    player_metadata_df['last_season'].astype('Int64')
)

# Save updated CSV
player_metadata_df.to_csv(
    '../data/processed/player_metadata.csv',
    index=False,
    encoding='utf-8-sig'
)

print("Player metadata updated successfully!")

Player metadata updated successfully!


In [70]:
player_metadata_df = pd.read_csv('..\data\processed\player_metadata.csv')

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\us\AppData\Local\Temp\ipykernel_19584\2746698074.py:1: SyntaxWarning: invalid escape sequence '\d'
  player_metadata_df = pd.read_csv('..\data\processed\player_metadata.csv')


In [71]:
player_metadata_df.head(5)

,player_id,canonical_name,full_name,nationality,batting_style,bowling_style,playing_role,batter,batter_runs,bowler,is_wicket,debut,last_season,team
0,3543,A Choudhary,Arpita Choudhary,India,Right-hand bat,No information available,Batter,A Choudhary,25.0,A Choudhary,5.0,2017.0,2017.0,Royal Challengers Bengaluru
1,3545,A Nehra,Ashish Nehra,India,Right-hand bat,Left-arm fast-medium,Bowler,A Nehra,41.0,A Nehra,121.0,2008.0,2017.0,Sunrisers Hyderabad
2,3547,B Kumar,Bhuvneshwar Kumar,India,Right-hand bat,Right-arm fast-medium,Bowler,B Kumar,354.0,B Kumar,243.0,2011.0,2026.0,Royal Challengers Bengaluru
3,3548,BCJ Cutting,Ben Cutting,Australia,Right-hand bat,Right-arm fast-medium,All-rounder,BCJ Cutting,238.0,BCJ Cutting,12.0,2014.0,2019.0,Mumbai Indians
4,3549,Bipul Sharma,Bipul Sharma,India,Left-hand bat,Left-arm orthodox,Bowler,Bipul Sharma,187.0,Bipul Sharma,22.0,2010.0,2017.0,Sunrisers Hyderabad


In [74]:
player_metadata_df = player_metadata_df[['player_id', 'canonical_name', 'full_name', 'nationality', 'batting_style', 'bowling_style', 'playing_role', 'batter_runs', 'is_wicket', 'debut', 'last_season', 'team']]

In [75]:
player_metadata_df.to_csv(
    '../data/processed/player_metadata.csv',
    index=False,
    encoding='utf-8-sig'
)

print("Player metadata updated successfully!")

Player metadata updated successfully!


In [76]:
player_metadata_df = pd.read_csv('..\data\processed\player_metadata.csv')

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\us\AppData\Local\Temp\ipykernel_19584\2746698074.py:1: SyntaxWarning: invalid escape sequence '\d'
  player_metadata_df = pd.read_csv('..\data\processed\player_metadata.csv')


In [77]:
player_metadata_df.head()

,player_id,canonical_name,full_name,nationality,batting_style,bowling_style,playing_role,batter_runs,is_wicket,debut,last_season,team
0,3543,A Choudhary,Arpita Choudhary,India,Right-hand bat,No information available,Batter,25.0,5.0,2017.0,2017.0,Royal Challengers Bengaluru
1,3545,A Nehra,Ashish Nehra,India,Right-hand bat,Left-arm fast-medium,Bowler,41.0,121.0,2008.0,2017.0,Sunrisers Hyderabad
2,3547,B Kumar,Bhuvneshwar Kumar,India,Right-hand bat,Right-arm fast-medium,Bowler,354.0,243.0,2011.0,2026.0,Royal Challengers Bengaluru
3,3548,BCJ Cutting,Ben Cutting,Australia,Right-hand bat,Right-arm fast-medium,All-rounder,238.0,12.0,2014.0,2019.0,Mumbai Indians
4,3549,Bipul Sharma,Bipul Sharma,India,Left-hand bat,Left-arm orthodox,Bowler,187.0,22.0,2010.0,2017.0,Sunrisers Hyderabad


In [78]:
# --------------------------------
# 1. Total runs
# --------------------------------
runs_df = (
    balls.groupby('batter', as_index=False)['batter_runs']
    .sum()
    .rename(columns={
        'batter': 'canonical_name',
        'batter_runs': 'runs'
    })
)

# --------------------------------
# 2. Balls faced
# Exclude wides; no-balls count as balls faced
# --------------------------------
valid_balls = balls[
    ~balls['extra_type']
    .fillna('')
    .str.lower()
    .str.contains('wide')
]

balls_faced_df = (
    valid_balls.groupby('batter')
    .size()
    .reset_index(name='balls_faced')
    .rename(columns={'batter': 'canonical_name'})
)

# --------------------------------
# 3. Dismissals
# Count the batter recorded in player_out
# --------------------------------
dismissed = balls[
    balls['player_out'].notna() &
    balls['player_out'].astype(str).str.strip().ne('')
]

dismissals_df = (
    dismissed.groupby('player_out')
    .size()
    .reset_index(name='dismissals')
    .rename(columns={'player_out': 'canonical_name'})
)

# --------------------------------
# 4. Combine batting statistics
# --------------------------------
batting_stats = (
    runs_df
    .merge(balls_faced_df, on='canonical_name', how='left')
    .merge(dismissals_df, on='canonical_name', how='left')
)

batting_stats[['balls_faced', 'dismissals']] = (
    batting_stats[['balls_faced', 'dismissals']]
    .fillna(0)
)

# --------------------------------
# 5. Calculate metrics
# --------------------------------
batting_stats['strike_rate'] = np.where(
    batting_stats['balls_faced'] > 0,
    batting_stats['runs'] / batting_stats['balls_faced'] * 100,
    np.nan
)

batting_stats['batting_average'] = np.where(
    batting_stats['dismissals'] > 0,
    batting_stats['runs'] / batting_stats['dismissals'],
    np.nan
)

batting_stats['strike_rate'] = batting_stats['strike_rate'].round(2)
batting_stats['batting_average'] = batting_stats['batting_average'].round(2)

In [79]:
# --------------------------------
# 6. Update player metadata
# --------------------------------
old_columns = [
    'runs', 'balls_faced', 'dismissals',
    'strike_rate', 'batting_average'
]

player_metadata_df = player_metadata_df.drop(
    columns=old_columns,
    errors='ignore'
)

player_metadata_df = player_metadata_df.merge(
    batting_stats,
    on='canonical_name',
    how='left'
)

# Fill missing counting statistics with zero
count_columns = ['runs', 'balls_faced', 'dismissals']

player_metadata_df[count_columns] = (
    player_metadata_df[count_columns].fillna(0)
)

# --------------------------------
# 7. Save updated CSV
# --------------------------------
player_metadata_df.to_csv(
    r'..\data\processed\player_metadata.csv',
    index=False,
    encoding='utf-8-sig'
)

print("Batting metrics saved successfully!")

display(
    player_metadata_df[
        ['full_name', 'runs', 'balls_faced',
         'dismissals', 'strike_rate', 'batting_average']
    ].sort_values('runs', ascending=False).head(10)
)


Batting metrics saved successfully!


,full_name,runs,balls_faced,dismissals,strike_rate,batting_average
112,Virat Kohli,9346.0,6930.0,231.0,134.86,40.46
42,Rohit Sharma,7331.0,5517.0,246.0,132.88,29.80
15,Shikhar Dhawan,6769.0,5326.0,193.0,127.09,35.07
7,David Warner,6567.0,4702.0,164.0,139.66,40.04
169,KL Rahul,5828.0,4188.0,128.0,139.16,45.53
63,Suresh Raina,5536.0,4046.0,171.0,136.83,32.37
38,MS Dhoni,5439.0,3957.0,142.0,137.45,38.30
25,Ajinkya Rahane,5367.0,4273.0,178.0,125.60,30.15
91,Sanju Samson,5181.0,3671.0,164.0,141.13,31.59
102,AB de Villiers,5181.0,3411.0,130.0,151.89,39.85


In [80]:

# ==========================================
# REMAINING BATTING METRICS
# ==========================================

# Exclude Super Overs from career batting statistics
regular_balls = balls[
    ~balls['is_super_over'].fillna(False).astype(bool)
].copy()


# 1. Matches played
matches_df = (
    regular_balls.groupby('batter')['match_id']
    .nunique()
    .reset_index(name='matches')
    .rename(columns={'batter': 'canonical_name'})
)


# 2. Highest score and innings-wise scores
innings_scores = (
    regular_balls.groupby(
        ['match_id', 'innings', 'batter'],
        as_index=False
    )['batter_runs']
    .sum()
    .rename(columns={
        'batter': 'canonical_name',
        'batter_runs': 'innings_runs'
    })
)

highest_score_df = (
    innings_scores.groupby('canonical_name')['innings_runs']
    .max()
    .reset_index(name='highest_score')
)


# 3. Hundreds and fifties
hundreds_df = (
    innings_scores.assign(
        hundred=(innings_scores['innings_runs'] >= 100).astype(int)
    )
    .groupby('canonical_name')['hundred']
    .sum()
    .reset_index(name='hundreds')
)

fifties_df = (
    innings_scores.assign(
        fifty=(
            (innings_scores['innings_runs'] >= 50) &
            (innings_scores['innings_runs'] < 100)
        ).astype(int)
    )
    .groupby('canonical_name')['fifty']
    .sum()
    .reset_index(name='fifties')
)


# 4. Fours and sixes
boundary_df = (
    regular_balls.groupby('batter', as_index=False)
    .agg(
        fours=('is_four', 'sum'),
        sixes=('is_six', 'sum')
    )
    .rename(columns={'batter': 'canonical_name'})
)


# 5. Combine all remaining metrics
remaining_stats = (
    matches_df
    .merge(highest_score_df, on='canonical_name', how='outer')
    .merge(hundreds_df, on='canonical_name', how='outer')
    .merge(fifties_df, on='canonical_name', how='outer')
    .merge(boundary_df, on='canonical_name', how='outer')
)


# 6. Merge into player metadata
new_columns = [
    'matches',
    'highest_score',
    'hundreds',
    'fifties',
    'fours',
    'sixes'
]

player_metadata_df = player_metadata_df.drop(
    columns=new_columns,
    errors='ignore'
)

player_metadata_df = player_metadata_df.merge(
    remaining_stats,
    on='canonical_name',
    how='left'
)

# Fill missing counting statistics
player_metadata_df[new_columns] = (
    player_metadata_df[new_columns].fillna(0)
)

player_metadata_df[new_columns] = (
    player_metadata_df[new_columns].astype(int)
)


# 7. Save updated CSV
player_metadata_df.to_csv(
    r'..\data\processed\player_metadata.csv',
    index=False,
    encoding='utf-8-sig'
)

print("Remaining batting metrics saved successfully!")

display(
    player_metadata_df[
        [
            'full_name',
            'runs',
            'strike_rate',
            'batting_average',
            'matches',
            'highest_score',
            'hundreds',
            'fifties',
            'fours',
            'sixes'
        ]
    ].sort_values('runs', ascending=False).head(15)
)


Remaining batting metrics saved successfully!


,full_name,runs,strike_rate,batting_average,matches,highest_score,hundreds,fifties,fours,sixes
112,Virat Kohli,9346.0,134.86,40.46,275,113,9,68,844,316
42,Rohit Sharma,7331.0,132.88,29.80,275,109,2,49,661,323
15,Shikhar Dhawan,6769.0,127.09,35.07,221,106,2,51,768,152
7,David Warner,6567.0,139.66,40.04,184,126,4,62,663,236
169,KL Rahul,5828.0,139.16,45.53,149,152,6,45,508,239
63,Suresh Raina,5536.0,136.83,32.37,200,100,1,39,506,203
38,MS Dhoni,5439.0,137.45,38.30,241,84,0,24,375,264
25,Ajinkya Rahane,5367.0,125.60,30.15,197,105,2,35,539,141
91,Sanju Samson,5181.0,141.13,31.59,185,119,5,27,432,243
102,AB de Villiers,5181.0,151.89,39.85,170,133,3,40,413,251


In [81]:
player_metadata_df

,player_id,canonical_name,full_name,nationality,batting_style,bowling_style,playing_role,batter_runs,is_wicket,debut,...,balls_faced,dismissals,strike_rate,batting_average,matches,highest_score,hundreds,fifties,fours,sixes
0,3543,A Choudhary,Arpita Choudhary,India,Right-hand bat,No information available,Batter,25.0,5.0,2017.0,...,20.0,1.0,125.00,25.00,3,15,0,0,1,1
1,3545,A Nehra,Ashish Nehra,India,Right-hand bat,Left-arm fast-medium,Bowler,41.0,121.0,2008.0,...,62.0,7.0,66.13,5.86,17,22,0,0,3,1
2,3547,B Kumar,Bhuvneshwar Kumar,India,Right-hand bat,Right-arm fast-medium,Bowler,354.0,243.0,2011.0,...,377.0,38.0,93.90,9.32,75,27,0,0,34,4
3,3548,BCJ Cutting,Ben Cutting,Australia,Right-hand bat,Right-arm fast-medium,All-rounder,238.0,12.0,2014.0,...,141.0,11.0,168.79,21.64,17,39,0,0,15,19
4,3549,Bipul Sharma,Bipul Sharma,India,Left-hand bat,Left-arm orthodox,Bowler,187.0,22.0,2010.0,...,123.0,8.0,152.03,23.38,17,35,0,0,11,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
799,35828,PSP Handscomb,Peter Handscomb,Australia,Right-hand bat,No information available,Wicket-keeper batter,6.0,NaN,2016.0,...,12.0,1.0,50.00,6.00,1,6,0,0,0,0
800,35941,SM Boland,Scott Boland,Australia,Right-hand bat,Right-arm fast-medium,Bowler,NaN,3.0,2016.0,...,0.0,0.0,NaN,NaN,0,0,0,0,0,0
801,36053,UT Khawaja,Usman Khawaja,Australia,Left-hand bat,Right-arm off-spin,Batter,127.0,NaN,2016.0,...,100.0,6.0,127.00,21.17,6,30,0,0,14,3
802,36198,F Behardien,Farhaan Behardien,South Africa,Right-hand bat,Right-arm medium-fast,All-rounder,14.0,NaN,2016.0,...,13.0,2.0,107.69,7.00,3,9,0,0,2,0


In [82]:
player_metadata_df.columns

Index(['player_id', 'canonical_name', 'full_name', 'nationality',
       'batting_style', 'bowling_style', 'playing_role', 'batter_runs',
       'is_wicket', 'debut', 'last_season', 'team', 'runs', 'balls_faced',
       'dismissals', 'strike_rate', 'batting_average', 'matches',
       'highest_score', 'hundreds', 'fifties', 'fours', 'sixes'],
      dtype='object')

In [83]:

# 1. Calculate batter runs in each innings
innings_scores = (
    regular_balls.groupby(
        ['match_id', 'innings', 'batter'],
        as_index=False
    )['batter_runs']
    .sum()
    .rename(columns={
        'batter': 'canonical_name',
        'batter_runs': 'innings_runs'
    })
)

# 2. Identify innings in which the batter was dismissed
dismissed_innings = (
    regular_balls[
        regular_balls['player_out'].notna()
    ]
    .groupby(['match_id', 'innings', 'player_out'])
    .size()
    .reset_index(name='dismissal_count')
    .rename(columns={'player_out': 'canonical_name'})
)

# 3. Merge dismissal information
innings_scores = innings_scores.merge(
    dismissed_innings,
    on=['match_id', 'innings', 'canonical_name'],
    how='left'
)

# 4. Identify not-out innings
innings_scores['not_out'] = (
    innings_scores['dismissal_count'].isna()
)

# 5. Select each player's highest score
highest_score_df = (
    innings_scores
    .sort_values(
        ['innings_runs', 'not_out'],
        ascending=[False, False]
    )
    .drop_duplicates('canonical_name')
    [['canonical_name', 'innings_runs', 'not_out']]
    .rename(columns={
        'innings_runs': 'highest_score'
    })
)

# 6. Create display format
highest_score_df['highest_score_display'] = (
    highest_score_df['highest_score'].astype(int).astype(str)
    + highest_score_df['not_out'].map({True: '*', False: ''})
)

display(highest_score_df.head(10))


,canonical_name,highest_score,not_out,highest_score_display
5298,CH Gayle,175,True,175*
0,BB McCullum,158,True,158*
18173,KL Rahul,152,True,152*
16942,Abhishek Sharma,141,False,141
14105,Q de Kock,140,True,140*
7528,AB de Villiers,133,True,133*
15375,Shubman Gill,129,False,129
10147,RR Pant,128,True,128*
2234,M Vijay,127,False,127
9144,DA Warner,126,False,126


In [84]:
player_metadata_df = player_metadata_df.drop(
    columns=[
        'highest_score',
        'highest_score_display',
        'not_out'
    ],
    errors='ignore'
)

player_metadata_df = player_metadata_df.merge(
    highest_score_df,
    on='canonical_name',
    how='left'
)

In [85]:
player_metadata_df

,player_id,canonical_name,full_name,nationality,batting_style,bowling_style,playing_role,batter_runs,is_wicket,debut,...,strike_rate,batting_average,matches,hundreds,fifties,fours,sixes,highest_score,not_out,highest_score_display
0,3543,A Choudhary,Arpita Choudhary,India,Right-hand bat,No information available,Batter,25.0,5.0,2017.0,...,125.00,25.00,3,0,0,1,1,15.0,True,15*
1,3545,A Nehra,Ashish Nehra,India,Right-hand bat,Left-arm fast-medium,Bowler,41.0,121.0,2008.0,...,66.13,5.86,17,0,0,3,1,22.0,True,22*
2,3547,B Kumar,Bhuvneshwar Kumar,India,Right-hand bat,Right-arm fast-medium,Bowler,354.0,243.0,2011.0,...,93.90,9.32,75,0,0,34,4,27.0,False,27
3,3548,BCJ Cutting,Ben Cutting,Australia,Right-hand bat,Right-arm fast-medium,All-rounder,238.0,12.0,2014.0,...,168.79,21.64,17,0,0,15,19,39.0,True,39*
4,3549,Bipul Sharma,Bipul Sharma,India,Left-hand bat,Left-arm orthodox,Bowler,187.0,22.0,2010.0,...,152.03,23.38,17,0,0,11,9,35.0,True,35*
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
799,35828,PSP Handscomb,Peter Handscomb,Australia,Right-hand bat,No information available,Wicket-keeper batter,6.0,NaN,2016.0,...,50.00,6.00,1,0,0,0,0,6.0,False,6
800,35941,SM Boland,Scott Boland,Australia,Right-hand bat,Right-arm fast-medium,Bowler,NaN,3.0,2016.0,...,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN
801,36053,UT Khawaja,Usman Khawaja,Australia,Left-hand bat,Right-arm off-spin,Batter,127.0,NaN,2016.0,...,127.00,21.17,6,0,0,14,3,30.0,False,30
802,36198,F Behardien,Farhaan Behardien,South Africa,Right-hand bat,Right-arm medium-fast,All-rounder,14.0,NaN,2016.0,...,107.69,7.00,3,0,0,2,0,9.0,True,9*


In [88]:
player_metadata_df.columns

Index(['player_id', 'canonical_name', 'full_name', 'nationality',
       'batting_style', 'bowling_style', 'playing_role', 'batter_runs',
       'is_wicket', 'debut', 'last_season', 'team', 'runs', 'balls_faced',
       'dismissals', 'strike_rate', 'batting_average', 'matches', 'hundreds',
       'fifties', 'fours', 'sixes', 'highest_score', 'not_out',
       'highest_score_display'],
      dtype='object')

In [89]:
 player_metadata_df[
        [
            'full_name',
            'runs',
            'strike_rate',
            'batting_average',
            'matches',
            'highest_score_display',
            'hundreds',
            'fifties',
            'fours',
            'sixes'
        ]
    ].sort_values('runs', ascending=False).head(15)

,full_name,runs,strike_rate,batting_average,matches,highest_score_display,hundreds,fifties,fours,sixes
112,Virat Kohli,9346.0,134.86,40.46,275,113*,9,68,844,316
42,Rohit Sharma,7331.0,132.88,29.80,275,109*,2,49,661,323
15,Shikhar Dhawan,6769.0,127.09,35.07,221,106*,2,51,768,152
7,David Warner,6567.0,139.66,40.04,184,126,4,62,663,236
169,KL Rahul,5828.0,139.16,45.53,149,152*,6,45,508,239
63,Suresh Raina,5536.0,136.83,32.37,200,100*,1,39,506,203
38,MS Dhoni,5439.0,137.45,38.30,241,84*,0,24,375,264
25,Ajinkya Rahane,5367.0,125.60,30.15,197,105*,2,35,539,141
91,Sanju Samson,5181.0,141.13,31.59,185,119,5,27,432,243
102,AB de Villiers,5181.0,151.89,39.85,170,133*,3,40,413,251


In [108]:
player_metadata_df.to_csv(
    r'..\data\processed\player_metadata.csv',
    index=False,
    encoding='utf-8-sig'
)

In [109]:
player_metadata_df = pd.read_csv('..\data\processed\player_metadata.csv')

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\us\AppData\Local\Temp\ipykernel_19584\2746698074.py:1: SyntaxWarning: invalid escape sequence '\d'
  player_metadata_df = pd.read_csv('..\data\processed\player_metadata.csv')


In [110]:
pd.set_option('display.max_columns', None)
player_metadata_df

,player_id,canonical_name,full_name,nationality,batting_style,bowling_style,playing_role,batter_runs,is_wicket,debut,last_season,team,runs,balls_faced,dismissals,strike_rate,batting_average,matches,hundreds,fifties,fours,sixes,highest_score,not_out,highest_score_display
0,3543,A Choudhary,Arpita Choudhary,India,Right-hand batter,No information available,Batter,25.0,5.0,2017.0,2017.0,Royal Challengers Bengaluru,25.0,20.0,1.0,125.00,25.00,3,0,0,1,1,15.0,True,15*
1,3545,A Nehra,Ashish Nehra,India,Right-hand batter,Left-arm fast-medium,Bowler,41.0,121.0,2008.0,2017.0,Sunrisers Hyderabad,41.0,62.0,7.0,66.13,5.86,17,0,0,3,1,22.0,True,22*
2,3547,B Kumar,Bhuvneshwar Kumar,India,Right-hand batter,Right-arm fast-medium,Bowler,354.0,243.0,2011.0,2026.0,Royal Challengers Bengaluru,354.0,377.0,38.0,93.90,9.32,75,0,0,34,4,27.0,False,27
3,3548,BCJ Cutting,Ben Cutting,Australia,Right-hand batter,Right-arm fast-medium,All-rounder,238.0,12.0,2014.0,2019.0,Mumbai Indians,238.0,141.0,11.0,168.79,21.64,17,0,0,15,19,39.0,True,39*
4,3549,Bipul Sharma,Bipul Sharma,India,Left-hand batter,Left-arm orthodox,Bowler,187.0,22.0,2010.0,2017.0,Sunrisers Hyderabad,187.0,123.0,8.0,152.03,23.38,17,0,0,11,9,35.0,True,35*
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
799,35828,PSP Handscomb,Peter Handscomb,Australia,Right-hand batter,No information available,Wicket-keeper batter,6.0,NaN,2016.0,2016.0,Rising Pune Supergiant,6.0,12.0,1.0,50.00,6.00,1,0,0,0,0,6.0,False,6
800,35941,SM Boland,Scott Boland,Australia,Right-hand batter,Right-arm fast-medium,Bowler,NaN,3.0,2016.0,2016.0,Rising Pune Supergiant,0.0,0.0,0.0,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN
801,36053,UT Khawaja,Usman Khawaja,Australia,Left-hand batter,Right-arm off-spin,Batter,127.0,NaN,2016.0,2016.0,Rising Pune Supergiant,127.0,100.0,6.0,127.00,21.17,6,0,0,14,3,30.0,False,30
802,36198,F Behardien,Farhaan Behardien,South Africa,Right-hand batter,Right-arm medium-fast,All-rounder,14.0,NaN,2016.0,2016.0,Punjab Kings,14.0,13.0,2.0,107.69,7.00,3,0,0,2,0,9.0,True,9*


In [94]:
player_metadata_df.columns

Index(['player_id', 'canonical_name', 'full_name', 'nationality',
       'batting_style', 'bowling_style', 'playing_role', 'batter_runs',
       'is_wicket', 'debut', 'last_season', 'team', 'runs', 'balls_faced',
       'dismissals', 'strike_rate', 'batting_average', 'matches', 'hundreds',
       'fifties', 'fours', 'sixes', 'highest_score', 'not_out',
       'highest_score_display'],
      dtype='object')

In [105]:
player_metadata_df['batting_style'] = player_metadata_df['batting_style'].str.replace('bat', 'batter')

In [111]:
player_metadata_df

,player_id,canonical_name,full_name,nationality,batting_style,bowling_style,playing_role,batter_runs,is_wicket,debut,last_season,team,runs,balls_faced,dismissals,strike_rate,batting_average,matches,hundreds,fifties,fours,sixes,highest_score,not_out,highest_score_display
0,3543,A Choudhary,Arpita Choudhary,India,Right-hand batter,No information available,Batter,25.0,5.0,2017.0,2017.0,Royal Challengers Bengaluru,25.0,20.0,1.0,125.00,25.00,3,0,0,1,1,15.0,True,15*
1,3545,A Nehra,Ashish Nehra,India,Right-hand batter,Left-arm fast-medium,Bowler,41.0,121.0,2008.0,2017.0,Sunrisers Hyderabad,41.0,62.0,7.0,66.13,5.86,17,0,0,3,1,22.0,True,22*
2,3547,B Kumar,Bhuvneshwar Kumar,India,Right-hand batter,Right-arm fast-medium,Bowler,354.0,243.0,2011.0,2026.0,Royal Challengers Bengaluru,354.0,377.0,38.0,93.90,9.32,75,0,0,34,4,27.0,False,27
3,3548,BCJ Cutting,Ben Cutting,Australia,Right-hand batter,Right-arm fast-medium,All-rounder,238.0,12.0,2014.0,2019.0,Mumbai Indians,238.0,141.0,11.0,168.79,21.64,17,0,0,15,19,39.0,True,39*
4,3549,Bipul Sharma,Bipul Sharma,India,Left-hand batter,Left-arm orthodox,Bowler,187.0,22.0,2010.0,2017.0,Sunrisers Hyderabad,187.0,123.0,8.0,152.03,23.38,17,0,0,11,9,35.0,True,35*
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
799,35828,PSP Handscomb,Peter Handscomb,Australia,Right-hand batter,No information available,Wicket-keeper batter,6.0,NaN,2016.0,2016.0,Rising Pune Supergiant,6.0,12.0,1.0,50.00,6.00,1,0,0,0,0,6.0,False,6
800,35941,SM Boland,Scott Boland,Australia,Right-hand batter,Right-arm fast-medium,Bowler,NaN,3.0,2016.0,2016.0,Rising Pune Supergiant,0.0,0.0,0.0,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN
801,36053,UT Khawaja,Usman Khawaja,Australia,Left-hand batter,Right-arm off-spin,Batter,127.0,NaN,2016.0,2016.0,Rising Pune Supergiant,127.0,100.0,6.0,127.00,21.17,6,0,0,14,3,30.0,False,30
802,36198,F Behardien,Farhaan Behardien,South Africa,Right-hand batter,Right-arm medium-fast,All-rounder,14.0,NaN,2016.0,2016.0,Punjab Kings,14.0,13.0,2.0,107.69,7.00,3,0,0,2,0,9.0,True,9*


In [112]:
player_metadata_df.to_csv(
    r'..\data\processed\player_metadata.csv',
    index=False,
    encoding='utf-8-sig'
)